# Macro Conditioning Study — PCA Relative Value Signals

**Objective:** Determine which macro factors (VIX, Gold, Oil, DXY, UST10Y) significantly
explain the daily P&L of our PCA-based relative value strategies across all regions.
Use the results to assign signal multipliers (2x / 1.5x / 1x) for regime-aware trading.

**Regression:**
$$\text{PnL}_t = \alpha + \beta_k \cdot \Delta\text{Factor}_{k,\, t-1} + \varepsilon_t$$

All factors are lagged by **1 business day** to preserve strict causality.
Factors are expressed as **log-changes** (not levels) to be stationary.
Standard errors are **HC3 heteroskedasticity-robust**.

---

In [1]:
import sys, os
# Make sure the AFP root is on the path so we can import MacroConditioning
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)   # run from project root so relative paths (data/, config.json) resolve

import warnings
import datetime as dt
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from MacroConditioning import (
    load_macro, run_full_pipeline,
    plot_beta_heatmap, plot_significance_bars,
    plot_bucket_sharpes, plot_rolling_betas,
    plot_multiplier_table,
    MACRO_LABELS,
)

print(f'Working directory: {os.getcwd()}')

Working directory: /Users/danyalsoomro/Desktop/MFE/Term4/AFP


## 0 · Configuration

Set the regions to include, the significance threshold for selecting conditioning factors,
and the rolling window for the stability analysis.

In [2]:
# ── Regions to analyse (comment out any you want to skip) ─────────────────────
REGIONS = [
    'Brazil',
    'Chile',
    'Colombia',
    'US',
    'EU',
    'India',
    # 'China',       # uncomment to include
    # 'FX Brazil',
    # 'FX Mexico',
]

T_THRESHOLD    = 2.0      # |t-stat| cutoff for factor significance
ROLLING_WINDOW = 63       # days for rolling beta estimation (~1 quarter)
LOOKBACK       = '5Y'     # PCA estimation lookback
BASE_DATE      = dt.date(2026, 1, 1)
OUTPUT_DIR     = 'reports/macro_conditioning'
CACHE_DIR      = 'cache'

## 1 · Macro Data Preview

In [3]:
macro_levels, macro_changes = load_macro('macro_asset.csv')

print(f'Macro data shape : {macro_levels.shape}')
print(f'Date range       : {macro_levels.index[0]}  →  {macro_levels.index[-1]}')
print(f'Factors          : {macro_levels.columns.tolist()}')
macro_levels.tail(5)

Macro data shape : (9389, 5)
Date range       : 1990-01-02  →  2025-12-31
Factors          : ['VIX', 'Gold', 'Oil', 'DXY', 'UST10Y']


,VIX,Gold,Oil,DXY,UST10Y
2025-12-25,13.47,4479.42,58.35,97.976,4.1335
2025-12-26,13.60,4533.21,56.74,98.022,4.1277
2025-12-29,14.20,4332.35,58.08,98.037,4.1102
2025-12-30,14.33,4339.49,57.95,98.238,4.1219
2025-12-31,14.95,4319.37,57.42,98.322,4.1670


In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

factors = macro_levels.columns.tolist()
n = len(factors)
fig = make_subplots(rows=n, cols=1, subplot_titles=factors, shared_xaxes=True,
                    vertical_spacing=0.04)

palette = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']
for i, factor in enumerate(factors, 1):
    fig.add_trace(
        go.Scatter(x=macro_levels.index, y=macro_levels[factor],
                   mode='lines', name=factor,
                   line=dict(color=palette[i-1], width=1.5)),
        row=i, col=1
    )

fig.update_layout(title='Macro Factor Levels (1990 – present)', height=900,
                  width=1000, template='plotly_white', showlegend=False)
fig.show()

## 2 · Run the Full Pipeline

This cell runs the pipeline for every region in `REGIONS`.
On first run each region's PCA residuals are computed and **cached** to `cache/`.
Subsequent runs load from cache (near-instant).

In [5]:
reg_summary, bkt_summary, mult_summary, region_data = run_full_pipeline(
    regions       = REGIONS,
    macro_path    = 'macro_asset.csv',
    lookback      = LOOKBACK,
    base_date     = BASE_DATE,
    t_threshold   = T_THRESHOLD,
    rolling_window= ROLLING_WINDOW,
    output_dir    = OUTPUT_DIR,
    cache_dir     = CACHE_DIR,
)

print(f'\nRegions processed : {list(region_data.keys())}')
print(f'Reg summary shape : {reg_summary.shape}')


  Brazil
  [Brazil] Computing PCA residuals (this may take a minute)…
Getting PCA residuals ...
Residuals successfully computed
Updating tradable tenors...
  [Brazil] Cached to cache/Brazil_data.pkl
  Done. Significant factors (|t|≥2.0): []

  Chile
  [Chile] Computing PCA residuals (this may take a minute)…
Getting PCA residuals ...
Residuals successfully computed
Updating tradable tenors...
  [Chile] Cached to cache/Chile_data.pkl
  Done. Significant factors (|t|≥2.0): ['DXY']

  Colombia
  [Colombia] Computing PCA residuals (this may take a minute)…
Getting PCA residuals ...
Residuals successfully computed
Updating tradable tenors...
  [Colombia] Cached to cache/Colombia_data.pkl
  Done. Significant factors (|t|≥2.0): []

  US
  [US] Loading from cache: cache/US_data.pkl
  Done. Significant factors (|t|≥2.0): []

  EU
  [EU] Computing PCA residuals (this may take a minute)…
Getting PCA residuals ...
Residuals successfully computed
Updating tradable tenors...
  [EU] Cached to cache/

## 3 · Regression Summary Table

Full β / T-Stat / R² table for every Region × Factor pair.
Rows with **|t| ≥ 2** are the candidates for signal conditioning.

In [6]:
def style_reg_table(df):
    """Highlight significant rows in green, near-significant in yellow."""
    sig   = df['T-Stat'].abs() >= 2.0
    near  = (df['T-Stat'].abs() >= 1.5) & ~sig
    styled = df.style\
        .format({'Beta': '{:.4f}', 'T-Stat': '{:.3f}',
                 'P-Value': '{:.4f}', 'R-Squared': '{:.4f}'})\
        .apply(lambda _: [
            'background-color: rgba(0,180,80,0.25)' if sig.iloc[i]
            else 'background-color: rgba(255,200,50,0.20)' if near.iloc[i]
            else ''
            for i in range(len(df))
        ], axis=0)\
        .set_caption('Green = |t|≥2 (significant), Yellow = |t|≥1.5 (borderline)')
    return styled

style_reg_table(reg_summary.sort_values('T-Stat', key=abs, ascending=False))

,Factor,Region,Beta,T-Stat,P-Value,R-Squared,N
8,DXY,Chile,-0.0117,-2.037,0.0417,0.0040,1923
16,Gold,US,-0.0028,-1.765,0.0776,0.0016,2637
19,UST10Y,US,0.0012,1.568,0.1168,0.0031,2637
4,UST10Y,Brazil,0.0016,1.492,0.1358,0.0046,2516
18,DXY,US,0.0053,1.416,0.1569,0.0013,2637
25,VIX,India,0.0003,1.218,0.2232,0.0011,2362
6,Gold,Chile,0.0030,1.196,0.2316,0.0014,1923
1,Gold,Brazil,-0.0019,-1.194,0.2323,0.0006,2516
10,VIX,Colombia,0.0006,1.066,0.2864,0.0007,2398
2,Oil,Brazil,-0.0007,-1.062,0.2884,0.0008,2516


## 4 · Beta Heatmap — Region × Factor

In [7]:
plot_beta_heatmap(reg_summary, value_col='Beta').show()

In [8]:
plot_beta_heatmap(reg_summary, value_col='T-Stat').show()

In [9]:
plot_beta_heatmap(reg_summary, value_col='R-Squared').show()

## 5 · Factor T-Statistics by Region

Bars crossing the red dashed line (|t| = 2) are statistically significant.
These are the factors eligible for signal conditioning.

In [10]:
plot_significance_bars(reg_summary, t_threshold=T_THRESHOLD).show()

## 6 · Conditional Sharpe — Bucket Analysis

Dates are split into **Low / Mid / High** terciles by the *lagged level* of each
macro factor. The Sharpe ratio within each bucket shows whether the strategy
performs better or worse in different macro regimes.

**Reading:** A clear monotonic pattern (Low < Mid < High, or vice versa) confirms
that the factor is a valid conditioning variable regardless of regression significance.

In [11]:
plot_bucket_sharpes(bkt_summary).show()

In [12]:
# Numeric bucket summary — sorted by |Sharpe Spread| descending
sharpe_cols = ['Region','Factor','Sharpe_Low','Sharpe_Mid','Sharpe_High','Sharpe_Spread']
sharpe_cols_present = [c for c in sharpe_cols if c in bkt_summary.columns]
bkt_display = bkt_summary[sharpe_cols_present].copy()
for c in ['Sharpe_Low','Sharpe_Mid','Sharpe_High','Sharpe_Spread']:
    if c in bkt_display.columns:
        bkt_display[c] = bkt_display[c].round(3)
bkt_display.sort_values('Sharpe_Spread', key=abs, ascending=False)

,Region,Factor,Sharpe_Low,Sharpe_Mid,Sharpe_High,Sharpe_Spread
5,Chile,VIX,2.689,2.102,0.441,-2.248
24,EU,UST10Y,0.468,1.367,2.277,1.808
26,India,Gold,1.871,1.887,0.301,-1.570
29,India,UST10Y,1.774,1.945,0.322,-1.452
0,Brazil,VIX,2.710,2.230,1.355,-1.355
13,Colombia,DXY,0.380,1.694,1.710,1.330
2,Brazil,Oil,1.677,1.695,2.938,1.261
19,US,UST10Y,1.327,3.184,0.141,-1.186
10,Colombia,VIX,0.687,1.251,1.837,1.150
28,India,DXY,1.984,1.342,1.032,-0.952


## 7 · Rolling Beta Stability

A factor is **operationally reliable** only if its beta is stable over time
(same sign, similar magnitude). Large sign-flips indicate the relationship
is regime-dependent and may not be exploitable in production.

Solid lines = significant in the full-sample regression.
Dotted lines = not significant (shown for comparison).

In [13]:
for country in region_data:
    plot_rolling_betas(country, region_data, t_threshold=T_THRESHOLD).show()

## 8 · Beta Sign-Consistency Check

For each region × factor, compute the **fraction of rolling windows where the beta
has the same sign as the full-sample beta**. A value > 65% indicates the relationship
is directionally reliable.

In [14]:
consistency_records = []

for country, data in region_data.items():
    rb  = data['rolling_betas']
    reg = data['regression']
    for factor in rb.columns:
        if factor not in reg.index:
            continue
        full_beta = reg.loc[factor, 'Beta']
        rolling   = rb[factor].dropna()
        if len(rolling) == 0:
            continue
        same_sign = (np.sign(rolling) == np.sign(full_beta)).mean()
        consistency_records.append({
            'Region':        country,
            'Factor':        factor,
            'Full Beta':     round(full_beta, 4),
            'T-Stat':        round(reg.loc[factor, 'T-Stat'], 3),
            'Sign Consist.': round(same_sign, 3),
            'Reliable':      same_sign >= 0.65,
        })

consistency_df = pd.DataFrame(consistency_records)
consistency_df.sort_values(['Region','Sign Consist.'], ascending=[True, False]).style\
    .applymap(lambda v: 'background-color: rgba(0,180,80,0.25)' if v is True
              else 'background-color: rgba(255,80,80,0.20)' if v is False else '',
              subset=['Reliable'])\
    .format({'Full Beta': '{:.4f}', 'T-Stat': '{:.3f}', 'Sign Consist.': '{:.1%}'})

,Region,Factor,Full Beta,T-Stat,Sign Consist.,Reliable
0,Brazil,VIX,-0.0002,-0.777,62.5%,False
1,Brazil,Gold,-0.0019,-1.194,55.1%,False
4,Brazil,UST10Y,0.0016,1.492,55.1%,False
2,Brazil,Oil,-0.0007,-1.062,53.0%,False
3,Brazil,DXY,0.0028,0.792,51.4%,False
5,Chile,VIX,-0.0002,-0.620,57.1%,False
8,Chile,DXY,-0.0117,-2.037,53.0%,False
9,Chile,UST10Y,-0.0005,-0.574,51.2%,False
7,Chile,Oil,-0.0002,-0.317,50.9%,False
6,Chile,Gold,0.0030,1.196,49.0%,False


## 9 · Multiplier Recommendations

Final output: **which factors to condition on, and at what scale.**

Factors are included only if they pass the |t| ≥ 2 threshold.
They are then ranked by |β| magnitude and bucketed:

| Bucket | Multiplier | Meaning |
|--------|-----------|----------|
| High   | **2.0×**  | Strategy is highly sensitive to this factor; amplify signal when regime is favorable |
| Mid    | **1.5×**  | Moderate sensitivity; moderate amplification |
| Low    | **1.0×**  | Weakly significant; keep signal unchanged |

**Direction** (+1 / -1) tells you which direction of the factor move is favorable.
The conditioning rule at time *t*:
$$m_{t} = \text{multiplier if } \text{sign}(\beta) \times \text{sign}(\Delta \text{Factor}_{t-1}) = +1 \text{, else } 1.0$$

In [15]:
plot_multiplier_table(mult_summary).show()

In [16]:
# Also as a DataFrame for easy programmatic access
if not mult_summary.empty:
    print(f'Total significant factor-region pairs: {len(mult_summary)}')
    display(mult_summary.sort_values(['Region','Multiplier'], ascending=[True, False]))
else:
    print(f'No factors passed the |t| ≥ {T_THRESHOLD} threshold. '
          f'Consider lowering T_THRESHOLD or extending the lookback period.')

Total significant factor-region pairs: 1


,Factor,Region,Beta,|Beta|,T-Stat,P-Value,R-Squared,Bucket,Multiplier,Direction
0,DXY,Chile,-0.011728,0.011728,-2.036887,0.041661,0.004021,High,2.0,-1.0


## 10 · Pairwise Correlation of Macro Factor Changes

Check for multicollinearity among factors. Highly correlated factors (|ρ| > 0.6)
should not both be used as independent conditioning variables.

In [17]:
import plotly.figure_factory as ff

# Compute correlation over the overlapping period with our data
# Use last 5 years for relevance
recent_changes = macro_changes.loc[macro_changes.index >= (macro_changes.index[-1] - pd.DateOffset(years=5).freqstr if False else macro_changes.index[-1])]
# Simpler: last 1260 rows ~5Y
recent_changes = macro_changes.iloc[-1260:].dropna()

corr = recent_changes.corr().round(3)

fig = ff.create_annotated_heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    annotation_text=corr.values.round(2).astype(str),
    colorscale='RdBu', zmid=0,
    showscale=True,
)
fig.update_layout(
    title='Pairwise Correlation of Macro Factor Log-Changes (last 5Y)',
    template='plotly_white', height=450, width=600,
)
fig.show()

## 11 · Sub-Period Stability

Split the sample into two equal halves and re-run the regression for each.
Factors where the beta sign is **consistent across both halves** are more likely
to be genuine relationships rather than full-sample artifacts.

In [18]:
from MacroConditioning import run_ts_regression

subperiod_records = []

for country, data in region_data.items():
    pnl   = data['pnl']
    mac_c = data['macro_changes']

    mid = pnl.index[len(pnl) // 2]
    for period_label, pnl_slice in [('First Half', pnl[pnl.index <= mid]),
                                    ('Second Half', pnl[pnl.index >  mid])]:
        mac_slice = mac_c.reindex(pnl_slice.index)
        reg_slice = run_ts_regression(pnl_slice, mac_slice)
        for factor, row in reg_slice.iterrows():
            subperiod_records.append({
                'Region':  country,
                'Factor':  factor,
                'Period':  period_label,
                'Beta':    round(row['Beta'], 4),
                'T-Stat':  round(row['T-Stat'], 3),
            })

subperiod_df = pd.DataFrame(subperiod_records)

# Pivot to show both halves side-by-side
pivot_beta = subperiod_df.pivot_table(
    index=['Region','Factor'], columns='Period', values='Beta'
).round(4)
pivot_tstat = subperiod_df.pivot_table(
    index=['Region','Factor'], columns='Period', values='T-Stat'
).round(3)

pivot_beta['Sign Consistent'] = (
    np.sign(pivot_beta.get('First Half', np.nan)) ==
    np.sign(pivot_beta.get('Second Half', np.nan))
)

print('Beta by sub-period:')
pivot_beta.style\
    .applymap(lambda v: 'background-color: rgba(0,180,80,0.25)' if v is True
              else 'background-color: rgba(255,80,80,0.20)' if v is False else '',
              subset=['Sign Consistent'])

Beta by sub-period:


## 12 · Export All Results

All tables are already saved to `reports/macro_conditioning/` by the pipeline.
This cell saves a combined Excel workbook for easy sharing.

In [19]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
excel_path = f'{OUTPUT_DIR}/MacroConditioning_Report.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    reg_summary.to_excel(writer, sheet_name='Regression Betas', index=False)
    bkt_summary.to_excel(writer, sheet_name='Bucket Sharpes',   index=False)
    if not mult_summary.empty:
        mult_summary.to_excel(writer, sheet_name='Multiplier Recs', index=False)
    consistency_df.to_excel(writer, sheet_name='Sign Consistency', index=False)
    pivot_beta.reset_index().to_excel(writer, sheet_name='Sub-Period Betas', index=False)

print(f'Report saved: {excel_path}')

Report saved: reports/macro_conditioning/MacroConditioning_Report.xlsx


---
## Summary / Decision Guide

Use this checklist to decide which factors to activate in production:

| Check | Criterion | Pass Threshold |
|-------|-----------|----------------|
| Statistical | |t-stat| ≥ 2.0 | ✅ Required |
| Directional | Sign consistency > 65% of rolling windows | ✅ Required |
| Sub-period | Same sign in both First and Second Half | ✅ Required |
| Economic | |Sharpe Spread| (High bucket − Low bucket) > 0.3 | ✅ Required |
| Orthogonality | |Correlation to other selected factors| < 0.6 | ✅ Required |

Factors satisfying **all five** checks receive the multiplier shown in Section 9.
Factors satisfying 3–4 checks are placed on a **watchlist** for future re-evaluation.
Factors satisfying ≤ 2 checks are **excluded**.